# 1 卷积层

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### 1.1 二维互相关运算

In [ ]:
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

### 1.2 学习卷积核

In [ ]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)
print('X:'); print(X)
print('Y:'); print(Y)

In [ ]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'batch {i + 1}, loss {l.sum():.3f}')

conv2d.weight.data.reshape((1, 2))

### 1.3 nn.Conv2d 使用

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(4, 3, 32, 32)
out = conv(x)
print('Input shape:', x.shape)
print('Output shape:', out.shape)
print('Number of parameters:', sum(p.numel() for p in conv.parameters()))

# 2 填充和步幅

In [ ]:
def comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
print('padding=1:', comp_conv2d(conv2d, X).shape)

In [ ]:
conv2d = nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2, 1))
print('不同高度宽度填充:', comp_conv2d(conv2d, X).shape)

In [ ]:
conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
print('stride=2:', comp_conv2d(conv2d, X).shape)

conv2d = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
print('stride=(3,4):', comp_conv2d(conv2d, X).shape)

In [ ]:
inp = torch.rand(1, 1, 28, 28)
convs = [
    ('k3_p0_s1', nn.Conv2d(1, 1, 3, padding=0, stride=1)),
    ('k3_p1_s1', nn.Conv2d(1, 1, 3, padding=1, stride=1)),
    ('k3_p1_s2', nn.Conv2d(1, 1, 3, padding=1, stride=2)),
    ('k5_p2_s2', nn.Conv2d(1, 1, 5, padding=2, stride=2)),
]
for name, conv in convs:
    out = conv(inp)
    print(f'{name}: {out.shape[-1]}')

# 3 多输入多输出通道

### 3.1 多输入通道

In [ ]:
def corr2d_multi_in(X, K):
    return sum(corr2d(x, k) for x, k in zip(X, K))

X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
                  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
corr2d_multi_in(X, K)

### 3.2 多输出通道

In [ ]:
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

K = torch.stack((K, K + 1, K + 2), 0)
print('K shape:', K.shape)
corr2d_multi_in_out(X, K)

In [ ]:
conv = nn.Conv2d(2, 3, kernel_size=3, padding=1)
x = torch.rand(1, 2, 5, 5)
out = conv(x)
print('Multi-in multi-out conv:', out.shape)

### 3.3 1x1 卷积

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6
print('1x1 conv result shape:', Y1.shape)

# 4 汇聚层

In [ ]:
def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i:i + p_h, j:j + p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i:i + p_h, j:j + p_w].mean()
    return Y

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
print('Max pool:', pool2d(X, (2, 2)))
print('Avg pool:', pool2d(X, (2, 2), 'avg'))

In [ ]:
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
print('X:'); print(X)

pool2d = nn.MaxPool2d(3)
print('MaxPool2d(3):', pool2d(X))

In [ ]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
print('MaxPool2d(3, padding=1, stride=2):', pool2d(X))

pool2d = nn.MaxPool2d((2, 3), padding=(1, 1), stride=(2, 3))
print('MaxPool2d((2,3), padding=(1,1), stride=(2,3)):', pool2d(X))

In [ ]:
avg_pool = nn.AvgPool2d(3, padding=1, stride=2)
print('AvgPool2d(3, padding=1, stride=2):', avg_pool(X))

X_multi = torch.rand(1, 3, 4, 4)
pool_multi = nn.MaxPool2d(2, stride=2)
print('Multi-channel pool:', pool_multi(X_multi).shape)

# 5 LeNet

### 5.1 LeNet 网络结构

In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.sigmoid(self.conv1(x))
        x = self.pool1(x)
        x = F.sigmoid(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = F.sigmoid(self.fc1(x))
        x = F.sigmoid(self.fc2(x))
        x = self.fc3(x)
        return x

net = LeNet()
print(net)

In [ ]:
X = torch.randn(1, 1, 28, 28)
for layer in net.children():
    X = layer(X)
    print(f'{layer.__class__.__name__}: {X.shape}')

### 5.2 加载 Fashion-MNIST 数据集

In [ ]:
batch_size = 256

def load_data_fashion_mnist(batch_size):
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
    ])
    mnist_train = torchvision.datasets.FashionMNIST(
        root='../data', train=True, transform=transform, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root='../data', train=False, transform=transform, download=True)
    train_iter = DataLoader(mnist_train, batch_size, shuffle=True)
    test_iter = DataLoader(mnist_test, batch_size, shuffle=False)
    return train_iter, test_iter

train_iter, test_iter = load_data_fashion_mnist(batch_size)

### 5.3 训练 LeNet

In [ ]:
def evaluate_accuracy(net, data_iter):
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in data_iter:
            y_hat = net(X)
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return correct / total

def train(net, train_iter, test_iter, num_epochs, lr):
    loss = nn.CrossEntropyLoss()
    trainer = torch.optim.SGD(net.parameters(), lr=lr)
    for epoch in range(num_epochs):
        net.train()
        total_loss, correct, total = 0.0, 0, 0
        for X, y in train_iter:
            trainer.zero_grad()
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            trainer.step()
            total_loss += l.item() * y.numel()
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.numel()
        train_acc = correct / total
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}, loss {total_loss / total:.4f}, '
              f'train acc {train_acc:.4f}, test acc {test_acc:.4f}')

train(net, train_iter, test_iter, num_epochs=5, lr=0.9)

### 5.4 测试结果展示

In [ ]:
def show_predictions(net, test_iter, num=10):
    net.eval()
    X, y = next(iter(test_iter))
    with torch.no_grad():
        y_hat = net(X)
    preds = y_hat.argmax(dim=1)
    labels_map = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
    fig, axes = plt.subplots(1, num, figsize=(12, 1.2))
    for i in range(num):
        axes[i].imshow(X[i].squeeze(), cmap='gray')
        axes[i].set_title(f'{labels_map[preds[i]]}')
        axes[i].axis('off')
    plt.show()

show_predictions(net, test_iter)

### 5.5 最终测试准确率

In [ ]:
test_acc = evaluate_accuracy(net, test_iter)
print(f'Test accuracy: {test_acc:.4f}')